# 03 — The shared instrument: a federated demand forecaster

**Goal (PROBLEM.md):** 138 hospitals jointly learn one demand forecaster **without sharing raw
data** — model weights travel, patient activity never does. The ladder below is the
instrument's honesty certificate: it says exactly what federation is and isn't worth.

**The rungs are alternatives, not a stack.** FedAvg / FedProx / FedAdam are three different
answers to "how do I merge 138 sets of client weights". Fine-tuning is a *stage* that composes
with any of them.

| rung | what it is |
|---|---|
| naive · seasonal-naive | no learning at all — the floor |
| **local-only** | every hospital alone — **the world without the federation** |
| FedAvg · FedProx · FedAdam | weights travel, data does not |
| + fine-tune | each hospital adapts the global model to itself |
| centralized | raw data pooled — **not an option**, privacy reference only |

**Protocol.** Selection on validation only; test evaluated once per rung. Fine-tuning uses the
*identical* budget and val-selected early stopping as local-only — the only difference is the
initialization, so the comparison isolates exactly one question: does starting from the
federation help a hospital?

**Pre-registered before running** (see `docs/GOALS.md`): the federation's advantage over
local-only should *grow* with model capacity and *shrink* with client size, because a small
hospital's ~30-450 rows cannot support a 36,721-parameter model while the federation's 62,280
can. Results are reported stratified by client size whichever way this lands.

In [ ]:
SMOKE = False    # True = plumbing check (few rounds). Set False for the real run.

ROUNDS = 20 if SMOKE else 500     # federated rounds
EPOCHS = 15 if SMOKE else 150     # local-only / centralized / fine-tune epoch budget
LOCAL_EPOCHS = 2                  # client epochs per federated round
HIDDEN, LAYERS, EMB = 128, 3, 16  # -> 36,721 parameters
DROPOUT, WD, LR, BATCH = 0.1, 1e-4, 1e-3, 256
PROX_MU = 0.1                     # FedProx proximal strength
SERVER_LR = 0.03                  # FedAdam server step
SEED = 0
print(f"SMOKE={SMOKE} | rounds={ROUNDS} | epochs={EPOCHS} | net={HIDDEN}x{LAYERS} emb={EMB}")

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

f = pd.read_csv("../data/processed/features_national.csv")
FEATS = ["lag1", "lag2", "lag3", "lag12", "backlog", "throughput", "month_sin", "month_cos"]
ser = ["provider_code", "specialty_code"]
specs = sorted(f["specialty_code"].unique())
f["spec"] = f["specialty_code"].map({s: i for i, s in enumerate(specs)})
clients = sorted(f["provider_code"].unique())
splits = {s: f[f["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"]}
te, va = splits["test"], splits["val"]
device = "cuda" if torch.cuda.is_available() else "cpu"

def tensors(rows):
    return (torch.tensor(rows[FEATS].values, dtype=torch.float32).to(device),
            torch.tensor(rows["spec"].values).to(device),
            torch.tensor(rows["y"].values, dtype=torch.float32).to(device))

# pre-materialize every tensor ONCE - 500 rounds x 138 clients must not re-convert dataframes
T = {s: tensors(splits[s]) for s in splits}
client_tr = {t: tensors(splits["train"][splits["train"]["provider_code"] == t]) for t in clients}
client_va = {t: tensors(va[va["provider_code"] == t]) for t in clients}
client_rows_va = {t: va[va["provider_code"] == t] for t in clients}
client_n = {t: len(client_tr[t][2]) for t in clients}
share = {t: client_n[t] / sum(client_n.values()) for t in clients}

class Net(nn.Module):
    def __init__(self, hidden=HIDDEN, layers=LAYERS, emb=EMB, dropout=DROPOUT):
        super().__init__()
        self.emb = nn.Embedding(len(specs), emb)
        dims, seq = [len(FEATS) + emb] + [hidden] * layers, []
        for a, b in zip(dims, dims[1:]):
            seq += [nn.Linear(a, b), nn.ReLU(), nn.Dropout(dropout)]
        self.mlp = nn.Sequential(*seq, nn.Linear(hidden, 1))
    def forward(self, x, s):   # residual on lag1 -> the model STARTS at the naive baseline
        return x[:, 0] + self.mlp(torch.cat([x, self.emb(s)], dim=1)).squeeze(-1)

def train_epochs(model, data, epochs, lr=LR, wd=WD, prox_ref=None, mu=0.0):
    x, s, y = data
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    model.train()
    for _ in range(epochs):
        perm = torch.randperm(len(y), device=x.device)
        for i in range(0, len(y), BATCH):
            j = perm[i:i + BATCH]
            opt.zero_grad()
            loss = nn.functional.l1_loss(model(x[j], s[j]), y[j])
            if prox_ref is not None:   # FedProx: keep local weights near the global reference
                loss = loss + (mu / 2) * sum(((p - g) ** 2).sum()
                                             for p, g in zip(model.parameters(), prox_ref))
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def predict(model, data, rows):
    model.eval()
    ylog = model(data[0], data[1]).cpu().numpy() * rows["sd"].values + rows["mu"].values
    return np.expm1(ylog)

PROG = "../data/processed/progress_national.log"
def note(msg):
    with open(PROG, "a") as fh:
        fh.write(f"{time.strftime('%H:%M:%S')}  {msg}\n")

print(f"device: {device} | clients: {len(clients)} | series: {f.groupby(ser).ngroups:,} | "
      f"params: {sum(p.numel() for p in Net().parameters()):,}")

## Metrics

MAE alone cannot compare a 35-client network with a 138-client one: averaging per-client MAE
over hospitals that range from 27 to 40,077 referrals/month mixes scales. So the primary
cross-run metrics are **scale-free** — sMAPE and skill-vs-naive — with MAE/RMSE/R² reported
alongside.

**R² is per-series, then averaged.** Pooled R² across 2,076 series with a 1,464x scale spread
would read ~0.99 purely because the model knows one hospital is bigger than another; it
measures nothing about forecast skill and is never reported here.

In [ ]:
def naive_pred(rows):
    return np.expm1(rows["lag1"].values * rows["sd"].values + rows["mu"].values)

NAIVE_TE = naive_pred(te)

def metrics(rows, pred, ref=None):
    a = rows["demand"].values
    err = pred - a
    key = pd.Series(list(zip(rows["provider_code"], rows["specialty_code"])))
    by_client = pd.Series(np.abs(err)).groupby(rows["provider_code"].values).mean()

    d = pd.DataFrame({"k": key.values, "a": a, "e2": err ** 2})
    ss_res = d.groupby("k")["e2"].sum()
    ss_tot = d.groupby("k")["a"].transform(lambda s: ((s - s.mean()) ** 2).sum()).groupby(d["k"]).first()
    r2 = (1 - ss_res / ss_tot.clip(lower=1e-9))

    ref = NAIVE_TE if ref is None else ref
    smape = np.mean(2 * np.abs(err) / np.clip(np.abs(pred) + np.abs(a), 1e-9, None)) * 100
    return {"mae_client": by_client.mean(), "mae_pooled": np.abs(err).mean(),
            "rmse": float(np.sqrt((err ** 2).mean())),
            "r2_series": float(r2.mean()), "r2_median": float(r2.median()),
            "skill_vs_naive": float(1 - (err ** 2).mean() / ((ref - a) ** 2).mean()),
            "smape": float(smape)}

def fit_with_val(data_tr, data_va, rows_va, epochs=EPOCHS, seed=SEED, init=None):
    torch.manual_seed(seed)
    model = Net().to(device)
    if init is not None:
        model.load_state_dict(init)
    best, best_m = {k: v.clone() for k, v in model.state_dict().items()}, np.inf
    for _ in range(epochs):
        train_epochs(model, data_tr, 1)
        m = np.abs(predict(model, data_va, rows_va) - rows_va["demand"].values).mean()
        if m < best_m:
            best_m, best = m, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best)
    return model

results, curves = {}, {}
for name, col in [("naive (last month)", "lag1"), ("seasonal-naive (t-12)", "lag12")]:
    results[name] = metrics(te, np.expm1(te[col].values * te["sd"].values + te["mu"].values))
    print(f"{name}: MAE {results[name]['mae_client']:.1f} | sMAPE {results[name]['smape']:.1f}%")

In [ ]:
# rung 1: every hospital alone - the world WITHOUT the federation
t0 = time.time()
pred = np.zeros(len(te))
local_models = {}
note("local-only start")
for i, t in enumerate(clients):
    m = fit_with_val(client_tr[t], client_va[t], client_rows_va[t])
    local_models[t] = {k: v.cpu() for k, v in m.state_dict().items()}
    mask = (te["provider_code"] == t).values
    if mask.any():
        rows = te[mask]
        pred[mask] = predict(m, tensors(rows), rows)
    if (i + 1) % 25 == 0:
        note(f"local-only client {i + 1}/{len(clients)}")
results["local-only"] = metrics(te, pred)
print(f"local-only: MAE {results['local-only']['mae_client']:.1f} | "
      f"sMAPE {results['local-only']['smape']:.1f}% | skill {results['local-only']['skill_vs_naive']:+.3f} "
      f"[{time.time() - t0:.0f}s]")

In [ ]:
# rungs 2-3: one trainer covering FedAvg / FedProx / FedAdam, each +- fine-tune
def fed_train(agg="avg", mu=0.0, rounds=ROUNDS, seed=SEED, tag="fed"):
    torch.manual_seed(seed)
    glob = Net().to(device)
    m_state = v_state = None                      # FedAdam server moments
    best, best_m, curve = None, np.inf, []
    for r in range(rounds):
        gsd = glob.state_dict()
        prox_ref = [p.detach().clone() for p in glob.parameters()] if mu > 0 else None
        avg = None
        for t in clients:                         # each client trains from global weights on ITS OWN rows
            local = Net().to(device)
            local.load_state_dict(gsd)
            train_epochs(local, client_tr[t], LOCAL_EPOCHS, prox_ref=prox_ref, mu=mu)
            sd = local.state_dict()
            avg = {k: share[t] * v for k, v in sd.items()} if avg is None \
                  else {k: avg[k] + share[t] * sd[k] for k in avg}
        if agg == "adam":                         # server-side Adam on the pseudo-gradient
            delta = {k: avg[k] - gsd[k] for k in avg}
            m_state = delta if m_state is None else {k: 0.9 * m_state[k] + 0.1 * delta[k] for k in delta}
            v_state = {k: delta[k] ** 2 for k in delta} if v_state is None \
                      else {k: 0.99 * v_state[k] + 0.01 * delta[k] ** 2 for k in delta}
            new = {k: gsd[k] + SERVER_LR * m_state[k] / (v_state[k].float().sqrt() + 1e-3) for k in delta}
        else:
            new = avg
        glob.load_state_dict(new)
        m = np.abs(predict(glob, T["val"], va) - va["demand"].values).mean()
        curve.append(float(m))
        if m < best_m:
            best_m, best = m, {k: v.clone() for k, v in new.items()}
        if r == 0 or (r + 1) % 25 == 0:
            note(f"{tag} round {r + 1}/{rounds}  val_mae {m:.2f}  best {best_m:.2f}")
    glob.load_state_dict(best)
    return glob, curve

def finetune(fed, tag="ft"):   # identical protocol to local-only; ONLY the init differs
    pred = np.zeros(len(te))
    models = {}
    for i, t in enumerate(clients):
        m = fit_with_val(client_tr[t], client_va[t], client_rows_va[t], init=fed.state_dict())
        models[t] = {k: v.cpu() for k, v in m.state_dict().items()}
        mask = (te["provider_code"] == t).values
        if mask.any():
            rows = te[mask]
            pred[mask] = predict(m, tensors(rows), rows)
        if (i + 1) % 25 == 0:
            note(f"{tag} fine-tune client {i + 1}/{len(clients)}")
    return pred, models

fed_models, ft_models = {}, {}
for label, agg, mu in [("FedAvg", "avg", 0.0), ("FedProx", "avg", PROX_MU), ("FedAdam", "adam", 0.0)]:
    t0 = time.time()
    fed, curve = fed_train(agg=agg, mu=mu, tag=label)
    curves[label] = curve
    results[label] = metrics(te, predict(fed, T["test"], te))
    fed_models[label] = {k: v.cpu() for k, v in fed.state_dict().items()}
    pred, models = finetune(fed, tag=label)
    results[f"{label} + fine-tune"] = metrics(te, pred)
    ft_models[label] = models
    print(f"{label}: MAE {results[label]['mae_client']:.1f} (best round "
          f"{int(np.argmin(curve)) + 1}/{len(curve)}) | +ft {results[label + ' + fine-tune']['mae_client']:.1f} "
          f"[{time.time() - t0:.0f}s]")

In [ ]:
# rung 4: centralized - raw data pooled. NOT an option in practice; a privacy-violating reference.
central = fit_with_val(T["train"], T["val"], va)
results["centralized (privacy violated)"] = metrics(te, predict(central, T["test"], te))

table = pd.DataFrame(results).T
table = table[["mae_client", "mae_pooled", "rmse", "smape", "skill_vs_naive", "r2_series", "r2_median"]]
table.columns = ["MAE/client", "MAE pooled", "RMSE", "sMAPE %", "skill vs naive", "R2 mean", "R2 median"]
print(table.sort_values("sMAPE %").round(3).to_string())

torch.save({"fed": fed_models, "fed_ft": ft_models, "local": local_models,
            "central": {k: v.cpu() for k, v in central.state_dict().items()},
            "specs": specs, "feats": FEATS,
            "cfg": {"hidden": HIDDEN, "layers": LAYERS, "emb": EMB, "dropout": DROPOUT,
                    "rounds": ROUNDS, "epochs": EPOCHS, "smoke": SMOKE}},
           "../data/processed/models_national.pt")
json.dump({"results": results, "curves": curves,
           "cfg": {"rounds": ROUNDS, "epochs": EPOCHS, "hidden": HIDDEN, "smoke": SMOKE}},
          open("../data/processed/results_national.json", "w"), indent=2)
print("\nsaved models_national.pt + results_national.json")